# Quarterly Data Transformation
#### This notebook transforms cleaned monthly indicators into a unified quarterly dataset for further econometric analysis.
### Methodology

The indicators used in this study differ in economic meaning and reporting logic. Therefore, they cannot be aggregated to quarterly frequency using one universal rule. To address this, two distinct aggregation schemes were applied depending on the nature of each indicator.

#### 1) Quarter-mean
This scheme is used for indicators that characterize average macroeconomic conditions during the quarter.

Applied to: `CPI`, `Policy_Rate`, `usd_uah`.

> Quarterly value = average of monthly values within the quarter


#### 2)  Quarter-end (date-adjusted) 
This scheme is used for banking indicators reported on the first day of the next month, where the date actually reflects the state at the end of the previous reporting period.

Applied to: `NPL`,`CAR`,`ROA`,`ROE`,`loans_to_clients`,`capital`.

> 1. Shift the reporting date one month back
> 2. Take the last observation of the quarter


*Example: `01.04` → corresponds to the end of Q1*

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
base_path = Path("../../data_processed")

cpi_path = base_path / "CPI_clean.xlsx"
exchange_path = base_path / "Exchange_Rate_clean.xlsx"
policy_rate_path = base_path / "Policy_Rate_cleaned.xlsx"
gdp_path = base_path / "GDP_clean.xlsx"
car_path = base_path / "CAR_clean.xlsx"
npl_path = base_path / "NPL_clean.xlsx"
banks_path = base_path / "Indicators_Banks_clean.xlsx"

output_path = base_path / "quarterly_data.xlsx"
output_path.parent.mkdir(parents=True, exist_ok=True)

In [3]:
# helper functions

def load_clean_file(path):
    df = pd.read_excel(path).copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

def to_quarter_label(series):
    return series.dt.to_period("Q").astype(str)


def quarter_mean(df, indicator_name):
    df = df.copy()
    df = df.sort_values("date")
    df = df.dropna(subset=["date"])
    df = df.set_index("date")

    df_q = df["value"].resample("Q").mean().reset_index()
    df_q["date"] = to_quarter_label(df_q["date"])
    df_q["indicator"] = indicator_name

    return df_q[["date", "value", "indicator"]]


def shift_one_month_back_then_last(df, indicator_name):
    df = df.copy()
    df = df.sort_values("date")
    df = df.dropna(subset=["date"])
    
    # reporting dates like 01.04 represent the state at the end of March
    df["date"] = df["date"] - pd.DateOffset(months=1)
    
    df = df.set_index("date")
    df_q = df["value"].resample("Q").last().reset_index()
    df_q["date"] = to_quarter_label(df_q["date"])
    df_q["indicator"] = indicator_name

    return df_q[["date", "value", "indicator"]]

def prepare_gdp_quarterly(df):
    df = df.copy()
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["date"] = pd.PeriodIndex(df["date"].astype(str), freq="Q").astype(str)
    df["indicator"] = "gdp_growth"
    return df[["date", "value", "indicator"]]


def tidy_to_wide(df):
    df_wide = df.pivot(index="date", columns="indicator", values="value").reset_index()
    df_wide.columns.name = None
    return df_wide

In [4]:
# load files

df_cpi = load_clean_file(cpi_path)
df_exchange = load_clean_file(exchange_path)
df_policy = load_clean_file(policy_rate_path)
df_gdp = pd.read_excel(gdp_path)
df_car = load_clean_file(car_path)
df_npl = load_clean_file(npl_path)
df_banks = load_clean_file(banks_path)

In [5]:
# check missing values

files_dict = {
    "CPI": df_cpi,
    "exchange_rate": df_exchange,
    "policy_rate": df_policy,
    "CAR": df_car,
    "NPL": df_npl,
    "Indicators_Banks": df_banks,
    "GDP":df_gdp
}

for name, df in files_dict.items():
    n_missing = df["value"].isna().sum()
    print(f"{name}: missing value count = {n_missing}")

CPI: missing value count = 0
exchange_rate: missing value count = 0
policy_rate: missing value count = 0
CAR: missing value count = 7
NPL: missing value count = 0
Indicators_Banks: missing value count = 0
GDP: missing value count = 0


## GDP

In [6]:
df_gdp_q = prepare_gdp_quarterly(df_gdp)
df_gdp_q.head()

,date,value,indicator
0,2015Q1,-0.160,gdp_growth
1,2015Q2,-0.145,gdp_growth
2,2015Q3,-0.070,gdp_growth
3,2015Q4,-0.024,gdp_growth
4,2016Q1,0.001,gdp_growth


## CPI, Policy Rate, Exchange Rate

In [7]:
df_cpi_q = quarter_mean(df_cpi, "CPI")
df_exchange_q = quarter_mean(df_exchange, "usd_uah")
df_policy_q = quarter_mean(df_policy, "policy_rate")

In [8]:
display(df_cpi_q.head(5))
display(df_exchange_q.head(5))
display(df_policy_q.head(5))

,date,value,indicator
0,2015Q4,43.300000,CPI
1,2016Q1,31.300000,CPI
2,2016Q2,8.066667,CPI
3,2016Q3,8.066667,CPI
4,2016Q4,12.300000,CPI


,date,value,indicator
0,2014Q4,15.768556,usd_uah
1,2015Q1,22.454521,usd_uah
2,2015Q2,21.036806,usd_uah
3,2015Q3,21.441587,usd_uah
4,2015Q4,23.596436,usd_uah


,date,value,indicator
0,2016Q1,22.000000,policy_rate
1,2016Q2,17.833333,policy_rate
2,2016Q3,15.333333,policy_rate
3,2016Q4,14.000000,policy_rate
4,2017Q1,14.000000,policy_rate


## CAR
### Inspect missing values in CAR 

In [9]:
for name, df in files_dict.items():
    missing_rows = df[df["value"].isna()]
    if not missing_rows.empty:
        print(f"\n{name} - missing rows:")
        print(missing_rows.head(20))


CAR - missing rows:
          date  value indicator
86  2022-03-01    NaN       CAR
87  2022-04-01    NaN       CAR
88  2022-05-01    NaN       CAR
116 2024-09-01    NaN       CAR
117 2024-10-01    NaN       CAR
118 2024-11-01    NaN       CAR
119 2024-12-01    NaN       CAR


In [10]:
period_check = df_car[
    (df_car["indicator"] == "CAR") & 
    (
        df_car["date"].between("2022-02-01", "2022-06-01") |
        df_car["date"].between("2024-08-01", "2025-01-01")
    )
]
display(period_check)

,date,value,indicator
85,2022-02-01,17.99,CAR
86,2022-03-01,NaN,CAR
87,2022-04-01,NaN,CAR
88,2022-05-01,NaN,CAR
89,2022-06-01,16.65,CAR
115,2024-08-01,19.85,CAR
116,2024-09-01,NaN,CAR
117,2024-10-01,NaN,CAR
118,2024-11-01,NaN,CAR
119,2024-12-01,NaN,CAR


* `2022Q1`: left as `NaN` due to the onset of the full-scale war in Ukraine, which disrupted reporting.
* `2024Q3`: `2024-09-01` is missing -> we use the previous month's value (`2024-08-01`) = `19.85`.

In [11]:
df_car_q = shift_one_month_back_then_last(df_car, "CAR")

# manually set value for 2022Q1
df_car_q.loc[df_car_q["date"] == "2022Q1", "value"] = np.nan

In [12]:
df_car_q[df_car_q["date"].isin(["2022Q1", "2024Q3"])]

,date,value,indicator
29,2022Q1,NaN,CAR
39,2024Q3,19.85,CAR


## NPL

In [13]:
df_npl_q = shift_one_month_back_then_last(df_npl, "NPL")
display(df_npl_q.head(5))

,date,value,indicator
0,2017Q1,55.106022,NPL
1,2017Q2,57.732746,NPL
2,2017Q3,56.435632,NPL
3,2017Q4,54.541296,NPL
4,2018Q1,56.446150,NPL


## Bank indicators: ROA, ROE, loans, capital

In [14]:
df_roa = df_banks[df_banks["indicator"] == "ROA"].copy()
df_roe = df_banks[df_banks["indicator"] == "ROE"].copy()
df_loans = df_banks[df_banks["indicator"] == "loans"].copy()
df_capital = df_banks[df_banks["indicator"] == "capital"].copy()

df_roa_q = shift_one_month_back_then_last(df_roa, "ROA")
df_roe_q = shift_one_month_back_then_last(df_roe, "ROE")
df_loans_q = shift_one_month_back_then_last(df_loans, "loans")
df_capital_q = shift_one_month_back_then_last(df_capital, "capital")

## Combine all files in long format

In [15]:
# combine all quarterly series in long format

df_quarterly_long = pd.concat(
    [
        df_cpi_q,
        df_exchange_q,
        df_policy_q,
        df_gdp_q,
        df_npl_q,
        df_car_q,
        df_roa_q,
        df_roe_q,
        df_loans_q,
        df_capital_q,
    ],
    ignore_index=True
)

df_quarterly_long = df_quarterly_long.sort_values(["date", "indicator"]).reset_index(drop=True)
df_quarterly_long.head(5)

,date,value,indicator
0,2014Q4,15.600000,CAR
1,2014Q4,15.768556,usd_uah
2,2015Q1,8.350000,CAR
3,2015Q1,-0.160000,gdp_growth
4,2015Q1,22.454521,usd_uah


In [17]:
# check observations by year and indicator

year_check = (
    df_quarterly_long.copy()
    .assign(year=lambda x: x["date"].str[:4].astype(int))
    .groupby(["year", "indicator"])["value"]
    .count()
    .reset_index(name="n")
    .pivot(index="year", columns="indicator", values="n")
    .fillna(0)
    .astype(int)
)

year_check

indicator,CAR,CPI,NPL,ROA,ROE,capital,gdp_growth,loans,policy_rate,usd_uah
year,,,,,,,,,,
2014,1,0,0,0,0,0,0,0,0,1
2015,4,1,0,1,1,1,4,1,0,4
2016,4,4,0,4,4,4,4,4,4,4
2017,4,4,4,4,4,4,4,4,4,4
2018,4,4,4,4,4,4,4,4,4,4
2019,4,4,4,4,4,4,4,4,4,4
2020,4,4,4,4,4,4,4,4,4,4
2021,4,4,4,4,4,4,4,4,4,4
2022,3,4,4,4,4,4,4,4,4,4


In [18]:
# convert to wide format

df_quarterly_wide = tidy_to_wide(df_quarterly_long)

column_order = [
    "date",
    "CPI",
    "usd_uah",
    "policy_rate",
    "gdp_growth",
    "NPL",
    "CAR",
    "ROA",
    "ROE",
    "loans",
    "capital",
]

df_quarterly_wide = df_quarterly_wide[column_order]
df_quarterly_wide = df_quarterly_wide.sort_values("date").reset_index(drop=True)

# keep balanced sample

df_final = df_quarterly_wide[
    (df_quarterly_wide["date"] >= "2017Q1") &
    (df_quarterly_wide["date"] <= "2024Q4") 
].reset_index(drop=True)

df_final.tail(5)

,date,CPI,usd_uah,policy_rate,gdp_growth,NPL,CAR,ROA,ROE,loans,capital
27,2023Q4,5.166667,36.907833,15.666667,0.052,37.351358,21.07,3.24,30.33,1024678.0,296044.0
28,2024Q1,4.066667,38.434567,14.833333,0.068,36.066158,20.44,5.45,50.30,1041062.0,339093.0
29,2024Q2,3.766667,40.235433,13.333333,0.040,34.570918,19.07,5.28,48.33,1087350.0,354533.0
30,2024Q3,7.166667,41.128533,13.000000,0.022,32.336924,19.85,5.14,45.62,1134872.0,391882.0
31,2024Q4,10.966667,41.637867,13.166667,-0.001,30.288271,17.35,2.94,25.52,1138032.0,368348.0


In [19]:
print(df_final.isna().sum())

date           0
CPI            0
usd_uah        0
policy_rate    0
gdp_growth     0
NPL            0
CAR            1
ROA            0
ROE            0
loans          0
capital        0
dtype: int64


In [20]:
# save 
df_final.to_excel(output_path, index=False)
print(f"saved to: {output_path}")

saved to: ..\..\data_processed\quarterly_data.xlsx


## Quality Checks 

In [21]:
print(df_final.shape)
print(df_final.isna().sum())

(32, 11)
date           0
CPI            0
usd_uah        0
policy_rate    0
gdp_growth     0
NPL            0
CAR            1
ROA            0
ROE            0
loans          0
capital        0
dtype: int64


In [22]:
print(df_final.describe())

             CPI    usd_uah  policy_rate  gdp_growth        NPL        CAR  \
count  32.000000  32.000000    32.000000   32.000000  32.000000  31.000000   
mean   10.675000  30.514574    14.562500   -0.012813  43.187108  18.930000   
std     6.230788   5.423958     5.636344    0.119099   9.594283   2.950476   
min     2.066667  24.239065     6.000000   -0.366000  27.064749  12.420000   
25%     6.683333  26.835942    10.500000   -0.012500  35.692348  16.725000   
50%     9.450000  27.799836    14.416667    0.027000  40.469490  19.070000   
75%    13.816667  36.568600    17.541667    0.039250  51.968664  21.330000   
max    26.566667  41.637867    25.000000    0.193000  57.732746  24.990000   

             ROA        ROE         loans        capital  
count  32.000000  32.000000  3.200000e+01      32.000000  
mean    2.820937  24.712187  1.047284e+06  222638.302733  
std     2.075302  19.048468  5.175388e+04   68854.571053  
min    -1.930000 -15.840000  9.605970e+05  140489.687451  
25

In [23]:
# Spot check 

expected = [
    {"date": "2017Q4", "indicator": "CPI","expected": 13.966},
    {"date": "2017Q4", "indicator": "CAR", "expected": 16.1},
    {"date": "2017Q4", "indicator": "NPL", "expected": 54.5413},
    {"date": "2022Q1", "indicator": "capital","expected": 208105},
    {"date": "2022Q1", "indicator": "CAR", "expected": 'NaN'},
    {"date": "2022Q1", "indicator": "NPL", "expected": 27.06475},
    {"date": "2022Q1", "indicator": "ROA", "expected": -0.03},
    {"date": "2024Q3", "indicator": "gdp_growth", "expected": 0.022} 
]

rows = []
for row in expected:
    actual = df_quarterly_wide.loc[
        df_quarterly_wide["date"] == row["date"], row["indicator"]
    ].values[0]
    rows.append({
        "date":      row["date"],
        "indicator": row["indicator"],
        "expected":  row["expected"],
        "actual":    round(actual, 4) if pd.notna(actual) else np.nan,
    })

display(pd.DataFrame(rows))

,date,indicator,expected,actual
0,2017Q4,CPI,13.966,13.9667
1,2017Q4,CAR,16.1,16.1000
2,2017Q4,NPL,54.5413,54.5413
3,2022Q1,capital,208105,208105.0000
4,2022Q1,CAR,NaN,NaN
5,2022Q1,NPL,27.06475,27.0647
6,2022Q1,ROA,-0.03,-0.0300
7,2024Q3,gdp_growth,0.022,0.0220
